# REAM: Router-weighted Expert Activation Merging

[![GitHub](https://img.shields.io/badge/GitHub-JackBinary%2FREAM-blue)](https://github.com/JackBinary/REAM)

Compress Mixture-of-Experts LLMs by merging expert groups. Based on [Boris Knyazev's blog post](https://bknyaz.github.io/blog/2026/moe/), forked from [CerebrasResearch/reap](https://github.com/CerebrasResearch/reap).

**Requirements:** A100 GPU (40GB+) for Qwen3-30B-A3B. Smaller MoE models may work on T4/L4.

## 1. Install

In [1]:
!git clone https://github.com/JackBinary/REAM.git
%cd REAM

Cloning into 'REAM'...
remote: Enumerating objects: 182, done.
remote: Counting objects: 100% (108/108), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 182 (delta 56), reused 72 (delta 40), pack-reused 74 (from 1)
Receiving objects: 100% (182/182), 4.49 MiB | 10.92 MiB/s, done.
Resolving deltas: 100% (68/68), done.
/content/REAM


In [2]:
!pip install uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.4/23.4 MB 128.5 MB/s eta 0:00:00


In [3]:
!git submodule init && git submodule update
!uv pip install --system --upgrade pip setuptools wheel
!VLLM_USE_PRECOMPILED=1 uv pip install --system --editable . -vv --torch-backend auto
# Upgrade transformers to support Qwen3.5 MoE (model_type "qwen3_5_moe" requires transformers >= 5.x)
!uv pip install --system "transformers @ git+https://github.com/huggingface/transformers.git"

Streaming output truncated to the last 5000 lines.
TRACE Removed file: /usr/local/lib/python3.12/dist-packages/torch/include/ATen/ops/zero_cuda_dispatch.h
TRACE Removed file: /usr/local/lib/python3.12/dist-packages/torch/include/ATen/ops/zero_meta_dispatch.h
TRACE Removed file: /usr/local/lib/python3.12/dist-packages/torch/include/ATen/ops/zero_native.h
TRACE Removed file: /usr/local/lib/python3.12/dist-packages/torch/include/ATen/ops/zero_ops.h
TRACE Removed file: /usr/local/lib/python3.12/dist-packages/torch/include/ATen/ops/zeros.h
TRACE Removed file: /usr/local/lib/python3.12/dist-packages/torch/include/ATen/ops/zeros_compositeexplicitautograd_dispatch.h
TRACE Removed file: /usr/local/lib/python3.12/dist-packages/torch/include/ATen/ops/zeros_like.h
TRACE Removed file: /usr/local/lib/python3.12/dist-packages/torch/include/ATen/ops/zeros_like_compositeexplicitautograd_dispatch.h
TRACE Removed file: /usr/local/lib/python3.12/dist-packages/torch/include/ATen/ops/zeros_like_compositeimp

## 2. Configuration

Edit these parameters to match your setup.

In [4]:
# ---- REAM Parameters ----
MODEL_NAME = "llmfan46/Qwen3.5-35B-A3B-heretic-v2"   # HuggingFace model ID
COMPRESSION_RATIO = 0.3125              # Fraction of experts to remove (0.25 = 128 -> 96)
MAX_CLUSTER_SIZE = 16                 # Max experts per merge group
SEED = 42

# ---- Evaluation flags ----
RUN_LM_EVAL = True
RUN_EVALPLUS = True
RUN_LIVE_CODE_BENCH = False
RUN_MATH = False

## 3. (Optional) HuggingFace Login

Required for gated models. Skip if your model is public.

In [8]:
# Uncomment to log in:
from google.colab import userdata

from huggingface_hub import login
login(userdata.get('HF_TOKEN'))

## 4. Run REAM

In [6]:
# Patch broken tokenizer_config.json for models that set tokenizer_class to "TokenizersBackend"
import json
from huggingface_hub import hf_hub_download

tok_cfg_path = hf_hub_download(MODEL_NAME, "tokenizer_config.json")
with open(tok_cfg_path) as f:
    tok_cfg = json.load(f)

if tok_cfg.get("tokenizer_class") == "TokenizersBackend":
    print(f"⚠ Patching invalid tokenizer_class 'TokenizersBackend' → removing field")
    del tok_cfg["tokenizer_class"]
    with open(tok_cfg_path, "w") as f:
        json.dump(tok_cfg, f, indent=2)
    print(f"  Patched: {tok_cfg_path}")
else:
    print(f"tokenizer_class is '{tok_cfg.get('tokenizer_class')}' — no patch needed")

tokenizer_config.json: 0.00B [00:00, ?B/s]

⚠ Patching invalid tokenizer_class 'TokenizersBackend' → removing field
  Patched: /root/.cache/huggingface/hub/models--llmfan46--Qwen3.5-35B-A3B-heretic-v2/snapshots/dd085a99db61758ca4c2f2271ea5a6b004f53b0b/tokenizer_config.json


In [ ]:
import subprocess, shlex, os, sys

short_name = MODEL_NAME.split("/")[-1]
server_log = f"ream_{short_name}_{SEED}.log"

cmd = [
    sys.executable, "-m", "ream.ream",
    "--compression-ratio", str(COMPRESSION_RATIO),
    "--model-name", MODEL_NAME,
    "--dataset-name", "ream_mixed",
    "--merge-method", "frequency_weighted_average",
    "--permute", "wm",
    "--cluster-method", "agglomerative",
    "--profile", "false",
    "--server_log_file_name", server_log,
    "--vllm-port", "8000",
    "--expert-sim", "router_logits",
    "--distance_measure", "cosine",
    "--frequency-penalty", "false",
    "--merged-model-dir-name", f"ream-{COMPRESSION_RATIO}",
    "--cluster-description", "ream",
    "--max-cluster-size", str(MAX_CLUSTER_SIZE),
    "--do-eval", str(RUN_LM_EVAL or RUN_EVALPLUS).lower(),
    "--run-lm-eval", str(RUN_LM_EVAL).lower(),
    "--run-evalplus", str(RUN_EVALPLUS).lower(),
    "--run-livecodebench", str(RUN_LIVE_CODE_BENCH).lower(),
    "--run-math", str(RUN_MATH).lower(),
    "--seed", str(SEED),
]

print("Running:", shlex.join(cmd))
print("="*60)

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="")

rc = process.wait()
if rc != 0:
    raise RuntimeError(f"REAM exited with code {rc}")
print("\nDone!")

Running: /usr/bin/python3 -m ream.ream --compression-ratio 0.3125 --model-name llmfan46/Qwen3.5-35B-A3B-heretic-v2 --dataset-name ream_mixed --merge-method frequency_weighted_average --permute wm --cluster-method agglomerative --profile false --server_log_file_name ream_Qwen3.5-35B-A3B-heretic-v2_42.log --vllm-port 8000 --expert-sim router_logits --distance_measure cosine --frequency-penalty false --merged-model-dir-name ream-0.3125 --cluster-description ream --max-cluster-size 16 --do-eval true --run-lm-eval true --run-evalplus true --run-livecodebench false --run-math false --seed 42
INFO 03-12 17:45:52 [__init__.py:235] Automatically detected platform cuda.
INFO:httpx:HTTP Request: HEAD https://huggingface.co/llmfan46/Qwen3.5-35B-A3B-heretic-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/llmfan46/Qwen3.5-35B-A3B-heretic-v2/dd085a99db61758ca4c2f2271ea5a6b004f53b0b/config.json "HTTP/1.1 200 OK

## 5. Inspect the Merged Model

After REAM finishes, the merged model is saved under `results/`. Check the output directory:

In [ ]:
import glob

result_dirs = sorted(glob.glob("results/**/ream-*", recursive=True))
for d in result_dirs:
    print(d)

## 6. (Optional) Upload to HuggingFace Hub

In [ ]:
# Uncomment and fill in to push your merged model:
#
# from huggingface_hub import HfApi
# api = HfApi()
# api.upload_folder(
#     folder_path="results/<YOUR_MODEL_DIR>",
#     repo_id="<YOUR_USERNAME>/<YOUR_REPO_NAME>",
#     repo_type="model",
# )